# 멤버 1번 소비 지표 및 행동 탐지 분석 (최종 수정본)

이 노트북은 과거(1~3월) 데이터와 오늘(4/1) 데이터를 비교하여 안정적 지표와 특이 행동을 심층 분석합니다.

In [2]:
import pandas as pd

# 1. 데이터 로드 및 전처리
df_past = pd.read_csv('../../../data/raw/csv/transactions_v1.csv')
df_today_full = pd.read_csv('./data_input_month.csv')

# 과거 데이터에서 멤버 1번만 추출
df_m1 = df_past[df_past['멤버 id'] == 1].copy()
df_m1['사용 시간'] = pd.to_datetime(df_m1['사용 시간'])
df_m1['date'] = df_m1['사용 시간'].dt.date

# 오늘 데이터 (2024-04-01) 추출
df_today_full['사용 시간'] = pd.to_datetime(df_today_full['사용 시간'])
df_today = df_today_full[df_today_full['사용 시간'].dt.strftime('%Y-%m-%d') == '2024-04-01'].copy()

# IQR 기반 클리핑 상하한선 설정
Q1 = df_m1['사용 금액'].quantile(0.25)
Q3 = df_m1['사용 금액'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = max(0, Q1 - 1.5 * IQR)
df_m1['사용 금액_clipped'] = df_m1['사용 금액'].clip(lower=lower_bound, upper=upper_bound)

print("✅ 데이터 로드 및 클리핑 설정 완료.")

✅ 데이터 로드 및 클리핑 설정 완료.


## [1] 클리핑 데이터 → 안정적 지표 분석
평소의 안정적인 소비 패턴(이상치 제외)과 오늘을 비교합니다.

In [3]:
# 과거 일평균(안정적) 계산
past_daily_stable_avg = df_m1.groupby('date')['사용 금액_clipped'].sum().mean()
today_total = df_today['사용 금액'].sum()

# 1. 평균 대비 및 증가율 계산
increase_rate = ((today_total - past_daily_stable_avg) / past_daily_stable_avg) * 100

print(f"[지표 1] 과거 일평균(안정): {past_daily_stable_avg:,.0f}원")
print(f"[지표 2] 오늘 총 지출액: {today_total:,.0f}원")
print(f"[지표 3] 평소 대비 지출 증가율: {increase_rate:.2f}%")

# 2. 카테고리 비율 비교
past_cat_ratio = df_m1.groupby('업종 카테고리')['사용 금액_clipped'].sum() / df_m1['사용 금액_clipped'].sum() * 100
today_cat_ratio = df_today.groupby('업종 카테고리')['사용 금액'].sum() / today_total * 100

cat_comparison = pd.DataFrame({
    '평소 비중(%)': past_cat_ratio,
    '오늘 비중(%)': today_cat_ratio
}).fillna(0)
cat_comparison['차이(pt)'] = cat_comparison['오늘 비중(%)'] - cat_comparison['평소 비중(%)']

print("\n--- 카테고리 비율 변화 ---")
display(cat_comparison.sort_values(by='차이(pt)', ascending=False))

[지표 1] 과거 일평균(안정): 51,014원
[지표 2] 오늘 총 지출액: 133,044원
[지표 3] 평소 대비 지출 증가율: 160.80%

--- 카테고리 비율 변화 ---


,평소 비중(%),오늘 비중(%),차이(pt)
업종 카테고리,,,
생활,1.470474,72.833048,71.362574
교통,4.741766,9.688524,4.946758
의료,8.093374,10.092902,1.999528
쇼핑,17.266950,0.000000,-17.266950
식비,68.427436,7.385527,-61.041910


## [2] 원본 데이터 → 행동 및 이상 탐지
원본 데이터를 기준으로 평소와 다른 특이 소비 행동을 탐지합니다.

In [4]:
# 1. 고액 소비 건 탐지 (과거 IQR 상한선 기준)
high_spending_items = df_today[df_today['사용 금액'] > upper_bound].copy()

# 2. 급증 소비 판단 (과거 원본 일평균 총액 대비)
past_daily_orig_avg = df_m1.groupby('date')['사용 금액'].sum().mean()
spike_ratio = today_total / past_daily_orig_avg

print(f"[탐지 1] 과거 원본 일평균(전체): {past_daily_orig_avg:,.0f}원")
if spike_ratio > 1.5:
    print(f"🚨 급증 소비 경보: 평소보다 {spike_ratio:.1f}배 높습니다!")
else:
    print("✅ 소비 규모가 평소 범위를 유지하고 있습니다.")

# 3. 특이 이벤트 내역 출력
print("\n--- [탐지 2] 오늘 발생한 특이 지출 내역 ---")
if not high_spending_items.empty:
    display(high_spending_items[['사용 시간', '결제 내역', '사용 금액', '업종 카테고리']])
else:
    print("오늘 발견된 특이 개별 지출 건이 없습니다.")

[탐지 1] 과거 원본 일평균(전체): 104,424원
✅ 소비 규모가 평소 범위를 유지하고 있습니다.

--- [탐지 2] 오늘 발생한 특이 지출 내역 ---


,사용 시간,결제 내역,사용 금액,업종 카테고리
2,2024-04-01 10:00:00,SKT통신비,65000,생활


## [3] 전날(3/31) 대비 소비 비교 분석
어제와 오늘의 지출 변화를 직접적으로 비교합니다.

In [5]:
# 전날(3/31) 데이터 추출
df_yesterday = df_m1[df_m1['date'].astype(str) == '2024-03-31'].copy()

yesterday_total = df_yesterday['사용 금액'].sum()
yesterday_count = len(df_yesterday)
today_count = len(df_today)

# 금액 및 건수 차이 계산
day_diff = today_total - yesterday_total
day_diff_rate = (day_diff / yesterday_total * 100) if yesterday_total > 0 else 0

print("--- 3/31(어제) vs 4/1(오늘) 비교 ---")
print(f"[지출액] 어제: {yesterday_total:,.0f}원 | 오늘: {today_total:,.0f}원 ({day_diff:+,.0f}원, {day_diff_rate:+.1f}%)")
print(f"[건수] 어제: {yesterday_count}건 | 오늘: {today_count}건 ({today_count - yesterday_count:+}건)")

# 주요 소비 카테고리 비교
if not df_yesterday.empty:
    yesterday_main_cat = df_yesterday.groupby('업종 카테고리')['사용 금액'].sum().idxmax()
    today_main_cat = df_today.groupby('업종 카테고리')['사용 금액'].sum().idxmax()
    print(f"[주 소비] 어제는 '{yesterday_main_cat}', 오늘은 '{today_main_cat}'에 가장 많이 썼습니다.")

--- 3/31(어제) vs 4/1(오늘) 비교 ---
[지출액] 어제: 48,531원 | 오늘: 133,044원 (+84,513원, +174.1%)
[건수] 어제: 6건 | 오늘: 9건 (+3건)
[주 소비] 어제는 '식비', 오늘은 '생활'에 가장 많이 썼습니다.


## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)
하루를 5개 구간으로 나누어 언제 소비가 집중되는지 분석합니다.

In [6]:
# 시간대 분류 함수 정의
def get_time_slot(hour):
    if 0 <= hour < 6:
        return '1.새벽(00-06)'
    if 6 <= hour < 11:
        return '2.오전(06-11)'
    if 11 <= hour < 17:
        return '3.점심/오후(11-17)'
    if 17 <= hour < 21:
        return '4.저녁(17-21)'
    return '5.밤/야식(21-24)'

# 오늘 데이터에 시간대 정보 추가
df_today['hour'] = df_today['사용 시간'].dt.hour
df_today['시간대'] = df_today['hour'].apply(get_time_slot)

# 과거 데이터(평균적 시간대 분포) 계산
df_m1['hour'] = df_m1['사용 시간'].dt.hour
df_m1['시간대'] = df_m1['hour'].apply(get_time_slot)
num_days = len(df_m1['date'].unique())
past_time_dist = df_m1.groupby('시간대')['사용 금액'].sum() / num_days

# 오늘 시간대별 합계 계산
today_time_dist = df_today.groupby('시간대')['사용 금액'].sum()

# 분석 결과 통합
time_analysis = pd.DataFrame({
    '오늘 지출액': today_time_dist,
    '평소 평균 지출액': past_time_dist
}).fillna(0)

print("--- 시간대별 소비 추세 분석 ---")
display(time_analysis.astype(int))

if not today_time_dist.empty:
    peak_slot = today_time_dist.idxmax()
    print(f"\n💡 오늘의 소비 피크 타임은 '{peak_slot}' 구간입니다.")

--- 시간대별 소비 추세 분석 ---


,오늘 지출액,평소 평균 지출액
시간대,,
2.오전(06-11),112895,9495
3.점심/오후(11-17),5471,68098
4.저녁(17-21),13428,16601
5.밤/야식(21-24),1250,10227



💡 오늘의 소비 피크 타임은 '2.오전(06-11)' 구간입니다.


- 과거 일평균과 오늘 지출액을 비교 (증가율 계산) 
- 카테고리별 비율 변화 (평소 비중, 오늘 비중) 
- 오늘 발생한 특이 지출 내역 
- 전날 대비 지출액 /  오늘 건수 / 주 소비 카테고리 비교 
- 시간별 소비 추세 분석 


In [7]:
def _format_currency(value: float | int) -> str:
    """숫자 금액을 보기 쉬운 원화 문자열로 변환한다."""
    return f"{float(value):,.0f}원"


def _build_category_change_lines(category_frame: pd.DataFrame, top_n: int = 3) -> list[str]:
    """카테고리 비중 변화 표를 마크다운 bullet 목록으로 요약한다."""
    if category_frame.empty:
        return ["### 카테고리 비중 변화", "- 카테고리 비중 변화 데이터가 없습니다."]

    sorted_frame = category_frame.sort_values(by='차이(pt)', ascending=False)
    increased = sorted_frame[sorted_frame['차이(pt)'] > 0].head(top_n)
    decreased = sorted_frame[sorted_frame['차이(pt)'] < 0].sort_values(by='차이(pt)').head(top_n)

    lines = ["### 카테고리 비중 변화", "#### 비중 증가 상위"]
    if increased.empty:
        lines.append("- 비중이 증가한 카테고리가 없습니다.")
    else:
        for category, row in increased.iterrows():
            lines.append(
                f"- {category}: 평소 {row['평소 비중(%)']:.1f}% -> 오늘 {row['오늘 비중(%)']:.1f}% ({row['차이(pt)']:+.1f}pt)"
            )

    lines.append("")
    lines.append("#### 비중 감소 상위")
    if decreased.empty:
        lines.append("- 비중이 감소한 카테고리가 없습니다.")
    else:
        for category, row in decreased.iterrows():
            lines.append(
                f"- {category}: 평소 {row['평소 비중(%)']:.1f}% -> 오늘 {row['오늘 비중(%)']:.1f}% ({row['차이(pt)']:+.1f}pt)"
            )
    return lines


def _build_high_spending_lines(high_spending_frame: pd.DataFrame) -> list[str]:
    """특이 지출 목록을 마크다운 bullet 목록으로 변환한다."""
    lines = ["### 특이 지출 내역"]
    if high_spending_frame.empty:
        lines.append("- 오늘 발견된 특이 개별 지출 건이 없습니다.")
        return lines

    sorted_frame = high_spending_frame.sort_values(by='사용 금액', ascending=False)
    for _, row in sorted_frame.iterrows():
        used_at = pd.to_datetime(row['사용 시간']).strftime('%Y-%m-%d %H:%M')
        lines.append(
            f"- {used_at} | {row['결제 내역']} | {_format_currency(row['사용 금액'])} | {row['업종 카테고리']}"
        )
    return lines


def _build_time_analysis_lines(time_frame: pd.DataFrame) -> list[str]:
    """시간대별 소비 분석 표를 마크다운 bullet 목록으로 변환한다."""
    lines = ["### 시간대별 세부 비교"]
    if time_frame.empty:
        lines.append("- 시간대별 소비 데이터가 없습니다.")
        return lines

    normalized_frame = time_frame.fillna(0).sort_index()
    for slot, row in normalized_frame.iterrows():
        today_amount = float(row['오늘 지출액'])
        usual_amount = float(row['평소 평균 지출액'])
        delta_amount = today_amount - usual_amount
        lines.append(
            f"- {slot}: 오늘 {_format_currency(today_amount)} / 평소 {_format_currency(usual_amount)} ({delta_amount:+,.0f}원)"
        )
    return lines


def build_daily_analysis_markdown(
    *,
    member_id: int,
    analysis_date: str,
    yesterday_label: str,
    past_daily_stable_avg: float,
    today_total: float,
    increase_rate: float,
    cat_comparison: pd.DataFrame,
    past_daily_orig_avg: float,
    spike_ratio: float,
    high_spending_items: pd.DataFrame,
    yesterday_total: float,
    yesterday_count: int,
    today_count: int,
    day_diff: float,
    day_diff_rate: float,
    yesterday_main_cat: str | None,
    today_main_cat: str | None,
    time_analysis: pd.DataFrame,
    peak_slot: str | None,
) -> str:
    """[1]~[4] 일일 소비 분석 결과를 하나의 마크다운 리포트로 합친다."""
    spike_message = (
        f"🚨 급증 소비 경보: 평소보다 {spike_ratio:.1f}배 높습니다!"
        if spike_ratio > 1.5
        else f"✅ 소비 규모가 평소 범위를 유지하고 있습니다. (평소 대비 {spike_ratio:.1f}배)"
    )

    lines = [
        f"# 멤버 {member_id}번 일일 소비 분석 종합 리포트",
        "",
        f"- 분석 기준일: {analysis_date}",
        f"- 비교 기준일: {yesterday_label}",
        "",
        "## [1] 클리핑 데이터 → 안정적 지표 분석",
        f"- 과거 일평균(안정): {_format_currency(past_daily_stable_avg)}",
        f"- 오늘 총 지출액: {_format_currency(today_total)}",
        f"- 평소 대비 지출 증가율: {increase_rate:+.2f}%",
        "",
        * _build_category_change_lines(cat_comparison),
        "",
        "## [2] 원본 데이터 → 행동 및 이상 탐지",
        f"- 과거 원본 일평균(전체): {_format_currency(past_daily_orig_avg)}",
        f"- 소비 규모 판단: {spike_message}",
        "",
        * _build_high_spending_lines(high_spending_items),
        "",
        f"## [3] 전날({yesterday_label}) 대비 소비 비교 분석",
        f"- 지출액 비교: {yesterday_label} {_format_currency(yesterday_total)} -> {analysis_date} {_format_currency(today_total)} ({day_diff:+,.0f}원, {day_diff_rate:+.1f}%)",
        f"- 결제 건수 비교: {yesterday_count}건 -> {today_count}건 ({today_count - yesterday_count:+}건)",
        f"- 주 소비 카테고리 변화: {yesterday_main_cat or '데이터 없음'} -> {today_main_cat or '데이터 없음'}",
        "",
        "## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)",
        f"- 오늘의 소비 피크 타임: {peak_slot or '확인 불가'}",
        "",
        * _build_time_analysis_lines(time_analysis),
    ]
    return '\n'.join(lines)


def render_notebook_daily_analysis_markdown(display_output: bool = True) -> str:
    """노트북에 계산된 [1]~[4] 분석 결과를 마크다운으로 렌더링하고 반환한다."""
    analysis_date = (
        df_today['사용 시간'].dt.strftime('%Y-%m-%d').iloc[0]
        if not df_today.empty
        else '오늘'
    )
    yesterday_label = str(df_yesterday['date'].iloc[0]) if not df_yesterday.empty else '전날'

    markdown_text = build_daily_analysis_markdown(
        member_id=1,
        analysis_date=analysis_date,
        yesterday_label=yesterday_label,
        past_daily_stable_avg=float(past_daily_stable_avg),
        today_total=float(today_total),
        increase_rate=float(increase_rate),
        cat_comparison=cat_comparison,
        past_daily_orig_avg=float(past_daily_orig_avg),
        spike_ratio=float(spike_ratio),
        high_spending_items=high_spending_items,
        yesterday_total=float(yesterday_total),
        yesterday_count=int(yesterday_count),
        today_count=int(today_count),
        day_diff=float(day_diff),
        day_diff_rate=float(day_diff_rate),
        yesterday_main_cat=globals().get('yesterday_main_cat'),
        today_main_cat=globals().get('today_main_cat'),
        time_analysis=time_analysis,
        peak_slot=globals().get('peak_slot'),
    )

    if display_output:
        try:
            from IPython.display import Markdown, display

            display(Markdown(markdown_text))
        except ImportError:
            print(markdown_text)
    return markdown_text


daily_analysis_markdown = render_notebook_daily_analysis_markdown(display_output=False)
assert '## [1] 클리핑 데이터 → 안정적 지표 분석' in daily_analysis_markdown
assert '## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)' in daily_analysis_markdown
assert 'SKT통신비' in daily_analysis_markdown

render_notebook_daily_analysis_markdown()


# 멤버 1번 일일 소비 분석 종합 리포트

- 분석 기준일: 2024-04-01
- 비교 기준일: 2024-03-31

## [1] 클리핑 데이터 → 안정적 지표 분석
- 과거 일평균(안정): 51,014원
- 오늘 총 지출액: 133,044원
- 평소 대비 지출 증가율: +160.80%

### 카테고리 비중 변화
#### 비중 증가 상위
- 생활: 평소 1.5% -> 오늘 72.8% (+71.4pt)
- 교통: 평소 4.7% -> 오늘 9.7% (+4.9pt)
- 의료: 평소 8.1% -> 오늘 10.1% (+2.0pt)

#### 비중 감소 상위
- 식비: 평소 68.4% -> 오늘 7.4% (-61.0pt)
- 쇼핑: 평소 17.3% -> 오늘 0.0% (-17.3pt)

## [2] 원본 데이터 → 행동 및 이상 탐지
- 과거 원본 일평균(전체): 104,424원
- 소비 규모 판단: ✅ 소비 규모가 평소 범위를 유지하고 있습니다. (평소 대비 1.3배)

### 특이 지출 내역
- 2024-04-01 10:00 | SKT통신비 | 65,000원 | 생활

## [3] 전날(2024-03-31) 대비 소비 비교 분석
- 지출액 비교: 2024-03-31 48,531원 -> 2024-04-01 133,044원 (+84,513원, +174.1%)
- 결제 건수 비교: 6건 -> 9건 (+3건)
- 주 소비 카테고리 변화: 식비 -> 생활

## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)
- 오늘의 소비 피크 타임: 2.오전(06-11)

### 시간대별 세부 비교
- 2.오전(06-11): 오늘 112,895원 / 평소 9,496원 (+103,399원)
- 3.점심/오후(11-17): 오늘 5,471원 / 평소 68,099원 (-62,628원)
- 4.저녁(17-21): 오늘 13,428원 / 평소 16,601원 (-3,173원)
- 5.밤/야식(21-24): 오늘 1,250원 / 평소 10,228원 (-8,978원)

'# 멤버 1번 일일 소비 분석 종합 리포트\n\n- 분석 기준일: 2024-04-01\n- 비교 기준일: 2024-03-31\n\n## [1] 클리핑 데이터 → 안정적 지표 분석\n- 과거 일평균(안정): 51,014원\n- 오늘 총 지출액: 133,044원\n- 평소 대비 지출 증가율: +160.80%\n\n### 카테고리 비중 변화\n#### 비중 증가 상위\n- 생활: 평소 1.5% -> 오늘 72.8% (+71.4pt)\n- 교통: 평소 4.7% -> 오늘 9.7% (+4.9pt)\n- 의료: 평소 8.1% -> 오늘 10.1% (+2.0pt)\n\n#### 비중 감소 상위\n- 식비: 평소 68.4% -> 오늘 7.4% (-61.0pt)\n- 쇼핑: 평소 17.3% -> 오늘 0.0% (-17.3pt)\n\n## [2] 원본 데이터 → 행동 및 이상 탐지\n- 과거 원본 일평균(전체): 104,424원\n- 소비 규모 판단: ✅ 소비 규모가 평소 범위를 유지하고 있습니다. (평소 대비 1.3배)\n\n### 특이 지출 내역\n- 2024-04-01 10:00 | SKT통신비 | 65,000원 | 생활\n\n## [3] 전날(2024-03-31) 대비 소비 비교 분석\n- 지출액 비교: 2024-03-31 48,531원 -> 2024-04-01 133,044원 (+84,513원, +174.1%)\n- 결제 건수 비교: 6건 -> 9건 (+3건)\n- 주 소비 카테고리 변화: 식비 -> 생활\n\n## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)\n- 오늘의 소비 피크 타임: 2.오전(06-11)\n\n### 시간대별 세부 비교\n- 2.오전(06-11): 오늘 112,895원 / 평소 9,496원 (+103,399원)\n- 3.점심/오후(11-17): 오늘 5,471원 / 평소 68,099원 (-62,628원)\n- 4.저녁(17-21): 오늘 13,428원 / 평소 16,601원 (-3,173원)\n- 5.밤/야식(21-24)

In [10]:
# 1. 지출 마찰력 (Frictionless Spending): 온라인/간편결제 비중
# 2. 지출 밀도 (Transaction Density): 결제 빈도(건수) vs 금액

# 온라인/간편결제 키워드
FRICTIONLESS_KEYWORDS = ['온라인', '간편결제', '앱결제', '배달']

# 마찰력 분석 (일일 데이터 df_today 사용)
is_frictionless = df_today['결제 방식 (온/오프라인)'].str.contains('|'.join(FRICTIONLESS_KEYWORDS), na=False)
df_fric = df_today[is_frictionless]

fric_total = float(df_fric['사용 금액'].sum())
fric_count = len(df_fric)
fric_ratio = (fric_total / today_total * 100) if today_total > 0 else 0.0

# 밀도 분석 (오늘의 결제 건수 및 건당 평균 금액)
today_count = len(df_today)
avg_per_swipe = today_total / today_count if today_count > 0 else 0

print("[일일 지출 마찰력 및 밀도 분석]")
print(f"  1. 지출 마찰력 (Pain of Paying)")
print(f"     - 온라인/간편결제: {fric_count}건 / {fric_total:,.0f}원")
print(f"     - 마찰력 없는 지출 비중: {fric_ratio:.1f}%")


print(f"\n  2. 지출 밀도 (Transaction Density)")
print(f"     - 오늘 결제 횟수: {today_count}회")
print(f"     - 1회 결제당 평균 금액: {avg_per_swipe:,.0f}원")


[일일 지출 마찰력 및 밀도 분석]
  1. 지출 마찰력 (Pain of Paying)
     - 온라인/간편결제: 1건 / 1,486원
     - 마찰력 없는 지출 비중: 1.1%

  2. 지출 밀도 (Transaction Density)
     - 오늘 결제 횟수: 9회
     - 1회 결제당 평균 금액: 14,783원
